In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t022'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory API traffic drop alert). It occurred between 06:21:20 and 06:28:54 (UTC+8) on April 13, 2026. The entity that triggered the alert is the operation `inventory::/api/v1/inventory/{productId}` within the APM domain. The user requests help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-04-13T06:21:20+08:00', 'end': '2026-04-13T06:28:54+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory接口流量下跌告警<alert_event event_id="5919adb97df530f82fed0ddb1249f942" trans_id="u80kouho9p4o95f6924vo7jsc0" rule_id="b7853e06-54b7-4409-9eea-9bb82fbfb252" rule_name="inventory接口流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across multiple entity sets, such as `node_ready_status`, `node_memory_usage_rate`, `node_cpu_usage_rate` (under `k8s.node`), and `error_rate`, `workload`, `slow` (under `apm.service.legacy` and `apm.operation`). To identify traffic-related KPIs, focus on metrics like `request_count` or similar, which are likely associated with `apm.operation` or `apm.service.legacy`.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
2100   node_memory_usage_rate            k8s.node
4060    node_not_ready_status            k8s.node
6160   node_pod_running_count            k8s.node
8260      node_cpu_usage_rate            k8s.node
...                       ...                 ...
78518                workload  apm.service.legacy
78530                    slow  apm.service.legacy
78545                   error  apm.service.legacy
88175              error_rate       apm.operation
88177              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data to include only 'apm.operation' and 'apm.service.legacy' entity sets
filtered_metrics_df = metrics_df[
    metrics_df['entity_set'].isin(['apm.operation', 'apm.service.legacy'])
]

# List all unique KPIs available within these entity sets
unique_kpis = filtered_metrics_df['metric'].unique()

unique_kpis

```
Out[3]:
```
Within the `apm.operation` and `apm.service.legacy` entity sets, the available KPIs include `error_count`, `slow_count`, `request_count`, `avg_request_latency_seconds`, `cpu`, `latency`, `mem`, `workload`, `slow`, `error`, and `error_rate`. For traffic-related analysis, the `request_count` metric is particularly relevant.

The original code execution output of IPython Kernel is also provided below for reference:

['error_count' 'slow_count' 'request_count' 'avg_request_latency_seconds'
 'cpu' 'latency' 'mem' 'workload' 'slow' 'error' 'error_rate']```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Filter data for the 'request_count' metric
request_count_df = filtered_metrics_df[filtered_metrics_df['metric'] == 'request_count']

# Convert 'time' column to numeric and calculate global P95 threshold
request_count_df['time'] = pd.to_numeric(request_count_df['time'], errors='coerce')
request_count_df['value'] = pd.to_numeric(request_count_df['value'], errors='coerce')
global_p95_threshold = request_count_df['value'].quantile(0.95)

# Define the time range in UTC+8
start_time = int(datetime.strptime('2026-04-13T06:21:20+08:00', '%Y-%m-%dT%H:%M:%S%z')
                 .astimezone(pytz.timezone('Asia/Shanghai')).timestamp() * 1_000_000)
end_time = int(datetime.strptime('2026-04-13T06:28:54+08:00', '%Y-%m-%dT%H:%M:%S%z')
               .astimezone(pytz.timezone('Asia/Shanghai')).timestamp() * 1_000_000)

# Filter data within the specified time range
filtered_time_range_df = request_count_df[
    (request_count_df['time'] >= start_time) & (request_count_df['time'] <= end_time)
]

global_p95_threshold, filtered_time_range_df

```
Out[4]:
```
The global P95 threshold for the `request_count` metric across all components is approximately 8521.9. Within the specified time range (06:21:20 to 06:28:54 UTC+8 on April 13, 2026), 488 data points for `request_count` were identified across various operations, such as `cart::POST /oteldemo.CartService/GetCart` and `shipping::oteldemo.ShippingService/GetQuote`. These metrics will help analyze traffic patterns and identify anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(np.float64(8521.899999999996),                    time domain     entity_set                         entity_id                                  entity_name         metric   value                            metric_set_id service
58365  1776032480000000    apm  apm.operation  7bb1b1512922edfdefac66f31c640a81     cart::POST /oteldemo.CartService/GetCart  request_count  5872.0  apm@metric_set@apm.metric.apm.operation    None
58366  1776032510000000    apm  apm.operation  7bb1b1512922edfdefac66f31c640a81     cart::POST /oteldemo.CartService/GetCart  request_count  6783.0  apm@metric_set@apm.metric.apm.operation    None
58367  1776032540000000    apm  apm.operation  7bb1b1512922edfdefac66f31c640a81     cart::POST /oteldemo.CartService/GetCart  request_count  9233.0  apm@metric_set@apm.metric.apm.operation    None
58368  1776032570000000    apm  apm.operation  7bb1b1512922edfdefac66f31c640a81     cart::POST /oteldemo.CartService/GetCart  request_count  7857.0  apm@metric_set@apm.metric.apm.operation    None
58369  1776032600000000    apm  apm.operation  7bb1b1512922edfdefac66f31c640a81     cart::POST /oteldemo.CartService/GetCart  request_count  5019.0  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                                          ...            ...     ...                                      ...     ...
62476  1776032810000000    apm  apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  request_count   594.0  apm@metric_set@apm.metric.apm.operation    None
62477  1776032840000000    apm  apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  request_count  2394.0  apm@metric_set@apm.metric.apm.operation    None
62478  1776032870000000    apm  apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  request_count  3479.0  apm@metric_set@apm.metric.apm.operation    None
62479  1776032900000000    apm  apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  request_count  3245.0  apm@metric_set@apm.metric.apm.operation    None
62480  1776032930000000    apm  apm.operation  d4bd2837257aa9b4e96fa92507b51d78  shipping::oteldemo.ShippingService/GetQuote  request_count  3196.0  apm@metric_set@apm.metric.apm.operation    None

[488 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Filter the data for the operation 'inventory::/api/v1/inventory/{productId}' within the specified time range
inventory_request_count_df = filtered_time_range_df[
    filtered_time_range_df['entity_name'] == 'inventory::/api/v1/inventory/{productId}'
]

# Identify anomalies where the request_count value falls significantly below the global P95 threshold
anomalies_df = inventory_request_count_df[
    inventory_request_count_df['value'] < global_p95_threshold
]

anomalies_df

```
Out[5]:
```
The `request_count` metric for the operation `inventory::/api/v1/inventory/{productId}` shows significant anomalies within the specified time range (06:21:20 to 06:28:54 UTC+8). The values consistently fall below the global P95 threshold of 8521.9, with the lowest value being 869. This indicates a substantial drop in traffic for this operation during the alert window.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                               entity_name         metric   value                            metric_set_id service
62325  1776032480000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3369.0  apm@metric_set@apm.metric.apm.operation    None
62326  1776032510000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3913.0  apm@metric_set@apm.metric.apm.operation    None
62327  1776032540000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  4406.0  apm@metric_set@apm.metric.apm.operation    None
62328  1776032570000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3486.0  apm@metric_set@apm.metric.apm.operation    None
62329  1776032600000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3105.0  apm@metric_set@apm.metric.apm.operation    None
62330  1776032630000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3504.0  apm@metric_set@apm.metric.apm.operation    None
62331  1776032660000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3263.0  apm@metric_set@apm.metric.apm.operation    None
62332  1776032690000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3042.0  apm@metric_set@apm.metric.apm.operation    None
62333  1776032720000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3475.0  apm@metric_set@apm.metric.apm.operation    None
62334  1776032750000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  2770.0  apm@metric_set@apm.metric.apm.operation    None
62335  1776032780000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count   869.0  apm@metric_set@apm.metric.apm.operation    None
62336  1776032810000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  1388.0  apm@metric_set@apm.metric.apm.operation    None
62337  1776032840000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3013.0  apm@metric_set@apm.metric.apm.operation    None
62338  1776032870000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3092.0  apm@metric_set@apm.metric.apm.operation    None
62339  1776032900000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3399.0  apm@metric_set@apm.metric.apm.operation    None
62340  1776032930000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3330.0  apm@metric_set@apm.metric.apm.operation    None```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces for the operation 'inventory::/api/v1/inventory/{productId}' during the alert window
filtered_traces_df = traces_df[
    (traces_df['spanName'] == 'inventory::/api/v1/inventory/{productId}') &
    (traces_df['startTime'] >= start_time * 1_000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= end_time * 1_000)        # Convert microseconds to nanoseconds
]

# Analyze upstream and downstream services contributing to the fault
upstream_downstream_services = filtered_traces_df[['traceId', 'spanName', 'serviceName', 'parentSpanId']].drop_duplicates()

upstream_downstream_services

```
Out[6]:
```
No traces were found for the operation `inventory::/api/v1/inventory/{productId}` during the alert window (06:21:20 to 06:28:54 UTC+8). This suggests that either the operation did not generate any trace data during this period or there was an issue with trace collection, making it difficult to analyze upstream or downstream services contributing to the fault.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanName, serviceName, parentSpanId]
Index: []```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Convert '_time_' column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs related to the 'inventory' operation during the alert window
inventory_logs_df = logs_df[
    (logs_df['_time_'] >= start_time) & 
    (logs_df['_time_'] <= end_time) & 
    (logs_df['content'].str.contains('inventory', case=False, na=False))
]

# Search for error messages or anomalies in the log content
error_logs_df = inventory_logs_df[
    inventory_logs_df['content'].str.contains('error|exception|fail|critical', case=False, na=False)
]

error_logs_df

```
Out[7]:
```
No logs containing error messages or anomalies were found for the `inventory` operation during the alert window (06:21:20 to 06:28:54 UTC+8). This indicates that there were no explicit error logs recorded for this operation during the incident period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```